### Using tools

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o-search-preview",
    web_search_options={},
    messages=[
        {
            "role": "user",
            "content": "What was a positive news story from today happened in China?",
        }
    ],
)

print(completion.choices[0].message.content)

On July 14, 2025, Chinese Vice President Han Zheng met with Indian External Affairs Minister S. Jaishankar in Beijing. During the meeting, Han emphasized the importance of continuing practical cooperation between China and India, highlighting the need for mutual respect to ensure stable bilateral relations. ([reuters.com](https://www.reuters.com/world/china/china-india-should-continue-practical-cooperation-chinese-vp-tells-indian-2025-07-14/?utm_source=openai))

Additionally, Chinese biotech shares have experienced a strong resurgence in 2025, driven by optimism around cancer treatment innovations and licensing deals with Western pharmaceutical companies. The Hang Seng Biotech Index has risen 61.8% this year, outpacing the broader market's performance. This growth marks a recovery from a previous downturn triggered by market volatility, regulatory changes, and geopolitical tensions. ([ft.com](https://www.ft.com/content/1beb84a6-71c1-494f-8a6c-b10a5da8b01b?utm_source=openai))


## Posit

In [3]:
from openai import OpenAI
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o-search-preview",
    web_search_options={
        "user_location": {
            "type": "approximate",
            "approximate": {
                "country": "GB",
                "city": "London",
                "region": "London",
            }
        },
    },
    messages=[{
        "role": "user",
        "content": "What are the best restaurants around Granary Square?",
    }],
)

print(completion.choices[0].message.content)

Granary Square in King's Cross is a vibrant hub offering a diverse array of dining options. Here are some notable restaurants in and around the square:

**[Granary Square Brasserie](https://www.granarysquarebrasserie.com?utm_source=openai)**  
**Open now · English · 3.7 (76 reviews)**  
_1 Granary Square (Granary Sq), London, Greater London, N1C 4AA_  
An all-day dining destination from The Ivy Collection, offering a menu of British and European classics in a stylish setting with both indoor and outdoor seating overlooking the fountains.

**[Caravan King's Cross](https://caravanandco.com/pages/kings-cross?utm_source=openai)**  
**Open now · Breakfast · $$ · 4.4 (1551 reviews)**  
_1 Granary Sq (Stable St), London, Greater London, N1C 4AA_  
Known for its specialty coffee and eclectic menu, Caravan serves dishes inspired by global cuisines in a relaxed, industrial-chic environment.

**[Dishoom King's Cross](http://www.dishoom.com/kings-cross?utm_source=openai)**  
**Open now · Indian · 

In [9]:
from openai import OpenAI
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o-search-preview",
    web_search_options={
        "search_context_size": "low",
    },
    messages=[{
        "role": "user",
        "content": "What movie won best picture in 2025?",
    }],
)

print(completion.choices[0].message.content)

At the 97th Academy Awards held on March 2, 2025, "Anora" won Best Picture. Directed by Sean Baker, the film also secured awards for Best Director, Best Actress (Mikey Madison), Best Original Screenplay, and Best Film Editing, totaling five Oscars. ([npr.org](https://www.npr.org/2025/03/02/nx-s1-5307165/oscars-2025-complete-winners-list?utm_source=openai))

"Anora" is a Brooklyn-set comedy-drama about an exotic dancer married to a Russian oligarch's son. ([apnews.com](https://apnews.com/article/796ad6e41d869cc793db1488f5a75f06?utm_source=openai))

The film was produced on a $6 million budget and earned close to $41 million globally. ([axios.com](https://www.axios.com/2025/03/03/oscars-academy-awards-anora-independent-films?utm_source=openai))


## 'Anora' Triumphs at 2025 Oscars:
- [Oscars 2025: 'Anora' wins Best Picture and Director among five Academy Awards](https://www.reuters.com/world/us/oscars-2025-academy-awards-who-is-nominated-what-time-does-it-start-2025-03-02/?utm_source=ope

### File search
https://platform.openai.com/docs/guides/tools-file-search

OpenAI 的文件搜索功能是为那些希望快速、简便地为 LLM 添加外部知识的企业和开发者设计的，它将 RAG 的复杂性抽象化，提供了一个开箱即用的解决方案。

In [10]:
import requests
from io import BytesIO
from openai import OpenAI

client = OpenAI()

def create_file(client, file_path):
    if file_path.startswith("http://") or file_path.startswith("https://"):
        # Download the file content from the URL
        response = requests.get(file_path)
        file_content = BytesIO(response.content)
        file_name = file_path.split("/")[-1]
        file_tuple = (file_name, file_content)
        result = client.files.create(
            file=file_tuple,
            purpose="assistants"
        )
    else:
        # Handle local file path
        with open(file_path, "rb") as file_content:
            result = client.files.create(
                file=file_content,
                purpose="assistants"
            )
    print(result.id)
    return result.id

# Replace with your own file path or URL
file_id = create_file(client, "https://cdn.openai.com/API/docs/deep_research_blog.pdf")

file-7x2U249grCRmETTp6xV6KF


In [11]:
vector_store = client.vector_stores.create(
    name="knowledge_base"
)
print(vector_store.id)

vs_6874d4e3d5d88191888edf37b40eab5a


In [12]:
result = client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file_id
)
print(result)

VectorStoreFile(id='file-7x2U249grCRmETTp6xV6KF', created_at=1752487188, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_6874d4e3d5d88191888edf37b40eab5a', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))


In [13]:
result = client.vector_stores.files.list(
    vector_store_id=vector_store.id
)
print(result)

SyncCursorPage[VectorStoreFile](data=[VectorStoreFile(id='file-7x2U249grCRmETTp6xV6KF', created_at=1752487188, last_error=None, object='vector_store.file', status='completed', usage_bytes=66539, vector_store_id='vs_6874d4e3d5d88191888edf37b40eab5a', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))], has_more=False, object='list', first_id='file-7x2U249grCRmETTp6xV6KF', last_id='file-7x2U249grCRmETTp6xV6KF')


In [14]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input="What is deep research by OpenAI?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": ["vs_6874d4e3d5d88191888edf37b40eab5a"]
    }]
)
print(response)

Response(id='resp_6874d57b249c819986a35cb154d414f70f54c703173042e8', created_at=1752487291.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFileSearchToolCall(id='fs_6874d57c74288199b47073716bb4d80c0f54c703173042e8', queries=['What is deep research by OpenAI?'], status='completed', type='file_search_call', results=None), ResponseOutputMessage(id='msg_6874d57ef108819997b61248a540efa80f54c703173042e8', content=[ResponseOutputText(annotations=[AnnotationFileCitation(file_id='file-7x2U249grCRmETTp6xV6KF', filename='deep_research_blog.pdf', index=544, type='file_citation'), AnnotationFileCitation(file_id='file-7x2U249grCRmETTp6xV6KF', filename='deep_research_blog.pdf', index=764, type='file_citation'), AnnotationFileCitation(file_id='file-7x2U249grCRmETTp6xV6KF', filename='deep_research_blog.pdf', index=934, type='file_citation'), AnnotationFileCitation(file_id='file-7x2U249grCRmETTp6xV6KF', filename='

#### Retrieval customization

In [16]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="What is deep research by OpenAI?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": ["vs_6874d4e3d5d88191888edf37b40eab5a"],
        "max_num_results": 2
    }]
)
print(response)

Response(id='resp_6874fa1e6844819a9ee99854930b7aaf08d02746e624db61', created_at=1752496670.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFileSearchToolCall(id='fs_6874fa1f51c0819a81142f11f7145eb908d02746e624db61', queries=['deep research by OpenAI', 'What is deep research OpenAI?'], status='completed', type='file_search_call', results=None), ResponseOutputMessage(id='msg_6874fa2115cc819a8faca2bb5e104a8708d02746e624db61', content=[ResponseOutputText(annotations=[AnnotationFileCitation(file_id='file-7x2U249grCRmETTp6xV6KF', filename='deep_research_blog.pdf', index=422, type='file_citation'), AnnotationFileCitation(file_id='file-7x2U249grCRmETTp6xV6KF', filename='deep_research_blog.pdf', index=1039, type='file_citation'), AnnotationFileCitation(file_id='file-7x2U249grCRmETTp6xV6KF', filename='deep_research_blog.pdf', index=1240, type='file_citation')], text='Deep Research by OpenAI is a new capabi

In [17]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="What is deep research by OpenAI?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": ["vs_6874d4e3d5d88191888edf37b40eab5a"]
    }],
    include=["file_search_call.results"]
)
print(response)

Response(id='resp_6874fbcb4378819bb15f0ce30f2374730336973202d202a8', created_at=1752497099.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFileSearchToolCall(id='fs_6874fbcc12f0819bb2f919b669c97be20336973202d202a8', queries=['What is deep research by OpenAI?'], status='completed', type='file_search_call', results=[Result(attributes={}, file_id='file-7x2U249grCRmETTp6xV6KF', filename='deep_research_blog.pdf', score=0.982, text="Introducing deep research | OpenAI\n\n\nFebruary 2, 2025 Release\n\nIntroducing deep research\nAn agent that uses reasoning to synthesize large amounts of\nonline information and complete multi-step research tasks\nfor you. Available to Pro users today, Plus and Team next.\n\nTry on ChatGPT\n\nListen to article 8:18 Share\n\n21/02/2025, 19:58 Introducing deep research | OpenAI\n\nhttps://openai.com/index/introducing-deep-research/ 1/38\n\nhttps://openai.com/research/index/r

In [20]:
from openai import OpenAI

# 初始化 OpenAI 客户端
# 确保你已经设置了 OpenAI API 密钥，例如通过环境变量 OPENAI_API_KEY
client = OpenAI()

# 假设你已经有了要删除的 vector_store_id
# 请将此处的 '<YOUR_VECTOR_STORE_ID>' 替换为你要删除的实际 vector store ID
vector_store_id_to_delete = "vs_6874d4e3d5d88191888edf37b40eab5a" # 替换为你要删除的实际 ID

try:
    # 删除指定的 vector store
    # delete() 方法会返回一个表示删除操作的对象，其中包含删除状态
    deleted_vector_store = client.vector_stores.delete(
        vector_store_id=vector_store_id_to_delete
    )

    # 打印删除结果
    print(f"Vector store '{vector_store_id_to_delete}' 删除成功。")
    print(f"删除状态: {deleted_vector_store.deleted}") # deleted 属性为 True 表示成功删除

except Exception as e:
    # 捕获并打印可能发生的错误
    print(f"删除 vector store '{vector_store_id_to_delete}' 时发生错误: {e}")

Vector store 'vs_6874d4e3d5d88191888edf37b40eab5a' 删除成功。
删除状态: True


### Remote MCP

In [7]:
from openai import OpenAI

client = OpenAI()

resp = client.responses.create(
    model="gpt-4.1",
    tools=[
        {
            "type": "mcp",
            "server_label": "deepwiki",
            "server_url": "https://mcp.deepwiki.com/mcp",
            "require_approval": "never",
        },
    ],
    input="What transport protocols are supported in the 2025-03-26 version of the MCP spec?",
)

print(resp.output_text)

As of the **2025-03-26** version of the **MCP (Mesh Configuration Protocol) specification**, the protocol explicitly supports the following **transport protocols**:

### Supported Transport Protocols

1. **gRPC**
2. **HTTP/2**

#### Details
- The spec formalizes that MCP messages **must be sent using gRPC**, typically running over **HTTP/2**.
- There are **no alternate transport protocols** such as HTTP/1.1 or raw TCP sockets specified for official support in this version.
- Secure transports (e.g., gRPC using TLS) are recommended but not mandated by the protocol spec itself.

#### Source
This is consistent with the [MCP specification history](https://github.com/istio/api/blob/master/mcp/spec/README.md) and version change notes. The **March 2025** revision did not introduce any additional transports beyond what was present in previous versions.

---

**Summary:**
> The 2025-03-26 version of the MCP spec supports transport over **gRPC (over HTTP/2)** only.

If you need the exact excerpt

In [9]:
from openai import OpenAI

client = OpenAI()

resp = client.responses.create(
    model="gpt-4.1",
    tools=[{
        "type": "mcp",
        "server_label": "deepwiki",
        "server_url": "https://mcp.deepwiki.com/mcp",
        "require_approval": "never",
        "allowed_tools": ["ask_question"],
    }],
    input="What transport protocols does the 2025-03-26 version of the MCP spec (modelcontextprotocol/modelcontextprotocol) support?",
)

print(resp.output_text)

The 2025-03-26 version of the Model Context Protocol (MCP) specification (modelcontextprotocol/modelcontextprotocol) supports the following transport protocols:

### 1. Standard Input/Output (stdio)
- Communication happens via standard input (`stdin`) and standard output (`stdout`) streams.
- This is suitable for local integrations and command-line tools.
- The client typically launches the MCP server as a subprocess, with each message exchanged as a line-delimited JSON-RPC 2.0 message.

### 2. Streamable HTTP
- Communication occurs over HTTP using HTTP POST requests for client-to-server messages.
- This replaces the earlier HTTP+SSE (Server-Sent Events) method, which is now deprecated.
- For HTTP transports, authorization with OAuth 2.1 is optional but recommended.

---

**Message Format:**  
In both transport types, all messages are formatted using JSON-RPC 2.0.

**Extensibility:**  
The spec also allows for the implementation of custom transports as long as they use JSON-RPC message

AttributeError: 'Response' object has no attribute 'mcp_approval_request'

### Image generation

In [19]:
from openai import OpenAI
import base64

client = OpenAI() 

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Generate an image of gray tabby cat hugging an otter with an orange scarf",
    tools=[{"type": "image_generation"}],
)

# Save the image to a file
image_data = [
    output.result
    for output in response.output
    if output.type == "image_generation_call"
]
    
if image_data:
    image_base64 = image_data[0]
    with open("otter.png", "wb") as f:
        f.write(base64.b64decode(image_base64))

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Your organization must be verified to use the model `gpt-image-1`. Please go to: https://platform.openai.com/settings/organization/general and click on Verify Organization. If you just verified, it can take up to 15 minutes for access to propagate.', 'type': 'invalid_request_error', 'param': None, 'code': None}}